[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shashvindu/gonu-jha-ai-master-jee/blob/main/SQL/sql_02_icc_points_table.ipynb) - run this notebook in your browser, nothing to install.

# SQL #2: Derive the ICC Points Table

**Gonu Jha AI Master Jee** · Interview prep notebook for the video *Derive the ICC Points Table* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. How to count matches when a team can appear in **two different columns**
2. `UNION ALL` to stack columns, `CASE WHEN` to build a 0/1 flag
3. `COUNT` vs `SUM` and `GROUP BY` to build the points table
4. **Two classic bugs, shown with real output:** `UNION` instead of `UNION ALL`, and `COUNT(win_flag)`
5. Extensions: points column and draws / no-results

In [1]:
# Everything runs on SQLite, which is built into Python - nothing to install.
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")

def sql(query):
    """Run a query and show the result as a table (a SQL NULL is shown as <NA>)."""
    return pd.read_sql_query(query, con).convert_dtypes()

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.50.4


## 1. Key definitions

| Term | Meaning |
|---|---|
| **`UNION ALL`** | Stacks the rows of two queries, **keeping duplicates** |
| **`UNION`** | Stacks rows and **removes duplicates** - dangerous for counting! |
| **`CASE WHEN ... THEN ... ELSE ... END`** | SQL's if/else, used here to make a 0/1 flag |
| **CTE** (`WITH name AS (...)`) | A named temporary result you can query like a table |
| **`GROUP BY`** | One output row per group; aggregates (`COUNT`, `SUM`) run inside each group |

> **Gonu Jha's story:** the kabaddi scorer counted wins from only ONE of his two notebooks (home teams), so half the matches vanished. Put both notebooks on one page first - that is `UNION ALL`.

## 2. The question
Input: `icc_world_cup(Team_1, Team_2, Winner)`, one row per match. Output: `team_name, matches_played, no_of_wins, no_of_losses`.

In [2]:
con.executescript("""
DROP TABLE IF EXISTS icc_world_cup;
CREATE TABLE icc_world_cup (Team_1 TEXT, Team_2 TEXT, Winner TEXT);
INSERT INTO icc_world_cup VALUES
  ('India', 'SL', 'India'), ('SL', 'Aus', 'Aus'), ('SA', 'Eng', 'Eng'),
  ('Eng', 'NZ', 'NZ'), ('Aus', 'India', 'India');
""")
sql("SELECT * FROM icc_world_cup")

,Team_1,Team_2,Winner
0,India,SL,India
1,SL,Aus,Aus
2,SA,Eng,Eng
3,Eng,NZ,NZ
4,Aus,India,India


## 3. Think before you write
1. A team can appear in `Team_1` **or** `Team_2` - so first stack both columns into one (`UNION ALL`).
2. For every (team, match) row add `win_flag` = 1 if the team is the `Winner`, else 0.
3. `GROUP BY` team: `COUNT(*)` = matches played, `SUM(win_flag)` = wins, the difference = losses.

## 4. Step 1: one row per team per match

In [3]:
all_teams = """
WITH all_teams AS (
    SELECT Team_1 AS team_name,
        CASE WHEN Team_1 = Winner THEN 1 ELSE 0 END AS win_flag
    FROM icc_world_cup
    UNION ALL
    SELECT Team_2,
        CASE WHEN Team_2 = Winner THEN 1 ELSE 0 END
    FROM icc_world_cup
)
"""
sql(all_teams + "SELECT * FROM all_teams")

,team_name,win_flag
0,India,1
1,SL,0
2,SA,0
3,Eng,0
4,Aus,0
5,SL,0
6,Aus,1
7,Eng,1
8,NZ,1
9,India,1


5 matches × 2 teams = **10 rows**. India appears twice (2 matches), both with win_flag 1.

## 5. Step 2: the points table

In [4]:
points_table = all_teams + """
SELECT team_name,
       COUNT(*) AS matches_played,
       SUM(win_flag) AS no_of_wins,
       COUNT(*) - SUM(win_flag) AS no_of_losses
FROM all_teams
GROUP BY team_name
ORDER BY no_of_wins DESC, team_name
"""
sql(points_table)

,team_name,matches_played,no_of_wins,no_of_losses
0,India,2,2,0
1,Aus,2,1,1
2,Eng,2,1,1
3,NZ,1,1,0
4,SA,1,0,1
5,SL,2,0,2


## 6. Bug 1: `UNION` instead of `UNION ALL`

In [5]:
sql(all_teams.replace("UNION ALL", "UNION") + "SELECT * FROM all_teams")

,team_name,win_flag
0,Aus,0
1,Aus,1
2,Eng,0
3,Eng,1
4,India,1
5,NZ,1
6,SA,0
7,SL,0


`UNION` removed the duplicate row (`India`, 1): India now has **1 match instead of 2**. Always use `UNION ALL` when you are going to count.

## 7. Bug 2: `COUNT(win_flag)` instead of `SUM(win_flag)`

In [6]:
sql(all_teams + """
SELECT team_name, COUNT(win_flag) AS wrong_wins, SUM(win_flag) AS correct_wins
FROM all_teams GROUP BY team_name ORDER BY team_name
""")

,team_name,wrong_wins,correct_wins
0,Aus,2,1
1,Eng,2,1
2,India,2,2
3,NZ,1,1
4,SA,1,0
5,SL,2,0


`COUNT` counts every non-NULL value - **including the zeros**. `SUM` adds the 1s, which is the number of wins.

## 8. Interview talking points
- Start by saying *"a team can appear in either column, so I'll first un-pivot both columns with UNION ALL"*.
- Explain why `UNION ALL` and not `UNION`.
- Ask: *"what about a draw or a no-result?"* - interviewers love this question.

### Common mistakes
- `UNION` instead of `UNION ALL`: India shows 1 match instead of 2
- `COUNT(win_flag)` counts the zeros too: use `SUM`
- Grouping only the `Winner` column drops teams with zero wins (SA and SL disappear)

## 9. Practice
**Task (from the video):** add a `points` column, where a win = 2 points.

In [7]:
# Solution
sql(all_teams + """
SELECT team_name,
       COUNT(*) AS matches_played,
       SUM(win_flag) AS no_of_wins,
       COUNT(*) - SUM(win_flag) AS no_of_losses,
       2 * SUM(win_flag) AS points
FROM all_teams
GROUP BY team_name
ORDER BY points DESC, team_name
""")

,team_name,matches_played,no_of_wins,no_of_losses,points
0,India,2,2,0,4
1,Aus,2,1,1,2
2,Eng,2,1,1,2
3,NZ,1,1,0,2
4,SA,1,0,1,0
5,SL,2,0,2,0


### Bonus: draws and no-results
If `Winner` can be `'Draw'` (or `NULL` for no result), give 1 point each and count draws separately:

In [8]:
con.execute("INSERT INTO icc_world_cup VALUES ('NZ', 'SA', 'Draw')")
sql("""
WITH all_teams AS (
    SELECT Team_1 AS team_name, Winner FROM icc_world_cup
    UNION ALL
    SELECT Team_2, Winner FROM icc_world_cup
)
SELECT team_name,
       COUNT(*) AS played,
       SUM(Winner = team_name) AS wins,
       SUM(Winner = 'Draw') AS draws,
       SUM(Winner <> team_name AND Winner <> 'Draw') AS losses,
       2 * SUM(Winner = team_name) + SUM(Winner = 'Draw') AS points
FROM all_teams
GROUP BY team_name
ORDER BY points DESC, team_name
""")

,team_name,played,wins,draws,losses,points
0,India,2,2,0,0,4
1,NZ,2,1,1,0,3
2,Aus,2,1,0,1,2
3,Eng,2,1,0,1,2
4,SA,2,0,1,1,1
5,SL,2,0,0,2,0


In SQLite (and MySQL) a comparison like `Winner = team_name` is already 1 or 0, so `SUM` of it counts the matches where it is true. In PostgreSQL / SQL Server write `SUM(CASE WHEN Winner = team_name THEN 1 ELSE 0 END)`.